# Детекция автоматизированного трафика

Задача — для каждой `cookie_id` вернуть `score` от 0 до 1: чем он выше, тем больше оснований считать трафик автоматизированным.

Я не использую события после окончания окна наблюдения и не пытаюсь определять бота по одному «подозрительному» полю. Основной сигнал здесь поведенческий: ритм событий, интенсивность действий, разнообразие объявлений и категорий, поисковая активность и pointer-поведение.

Пайплайн простой: загрузка данных → фильтрация по временному окну → признаки на уровне cookie → честная временная валидация → CatBoost → ансамбль из нескольких seed → `submission.csv`.


In [ ]:
# 1. Окружение и импорты

from pathlib import Path
from importlib.metadata import PackageNotFoundError, version as package_version
import hashlib
import importlib.util
import random
import re
import subprocess
import sys
import time
import warnings
import zipfile

# CatBoost не установлен в чистом Colab, поэтому ставим его автоматически.
# scikit-learn обычно уже есть, но на всякий случай проверяем и его.
def ensure_package(import_name, distribution_name, required_version):
    installed = importlib.util.find_spec(import_name) is not None

    current_version = None
    if installed:
        try:
            current_version = package_version(distribution_name)
        except PackageNotFoundError:
            current_version = None

    if current_version != required_version:
        requirement = f"{distribution_name}=={required_version}"
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", requirement]
        )


ensure_package("catboost", "catboost", "1.2.10")
ensure_package("sklearn", "scikit-learn", "1.6.1")

import numpy as np
import pandas as pd
import catboost
import sklearn

from IPython.display import display
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# В Colab можно открыть стандартное окно загрузки файла.
try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

# В локальном Jupyter пробуем открыть обычный системный диалог выбора файла.
# На удалённом Jupyter GUI может быть недоступен — этот вариант просто пропустится.
try:
    import tkinter as tk
    from tkinter import filedialog
except Exception:
    tk = None
    filedialog = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings("ignore")

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("CatBoost:", catboost.__version__)
print("scikit-learn:", sklearn.__version__)


Python: 3.13.15
NumPy: 2.1.3
Pandas: 2.2.3
CatBoost: 1.2.10
scikit-learn: 1.6.1


In [ ]:
# 2. Поиск, выбор и распаковка данных

PROJECT_DIR = Path.cwd()

# Если нужно, здесь можно явно указать путь к архиву.
# Обычно это не требуется: notebook сначала ищет данные и архив сам.
ARCHIVE_PATH = None

REQUIRED_DATA_FILES = (
    Path("data/train.csv"),
    Path("data/test.csv"),
    Path("data/events.csv.gz"),
    Path("sample_submission.csv"),
    Path("metric.py"),
)


def is_dataset_root(path):
    path = Path(path)
    return all((path / relative_path).exists() for relative_path in REQUIRED_DATA_FILES)


def find_dataset_root(base_dir):
    base_dir = Path(base_dir)

    # Сначала проверяем самые ожидаемые места без рекурсивного обхода.
    direct_candidates = [
        base_dir,
        base_dir / "bot_detection_challenge",
    ]

    for candidate in direct_candidates:
        if is_dataset_root(candidate):
            return candidate

    # Если папка называется иначе, ищем train.csv и проверяем соседние файлы.
    for train_file in base_dir.glob("*/data/train.csv"):
        candidate = train_file.parent.parent
        if is_dataset_root(candidate):
            return candidate

    return None


def choose_archive(base_dir):
    base_dir = Path(base_dir)

    if ARCHIVE_PATH is not None:
        archive = Path(ARCHIVE_PATH).expanduser().resolve()
        if not archive.exists():
            raise FileNotFoundError(f"Указанный архив не найден: {archive}")
        return archive

    preferred = base_dir / "bot_detection_challenge.zip"
    if preferred.exists():
        return preferred

    zip_files = sorted(base_dir.glob("*.zip"))
    if len(zip_files) == 1:
        return zip_files[0]

    if colab_files is not None:
        print("Архив с данными не найден автоматически. Выберите zip-файл в открывшемся окне.")
        uploaded = colab_files.upload()
        if not uploaded:
            raise RuntimeError("Файл не был выбран.")
        uploaded_name = next(iter(uploaded))
        return base_dir / uploaded_name

    if tk is not None and filedialog is not None:
        try:
            root = tk.Tk()
            root.withdraw()
            root.attributes("-topmost", True)
            selected = filedialog.askopenfilename(
                title="Выберите архив с данными",
                filetypes=[("ZIP archive", "*.zip"), ("All files", "*.*")],
            )
            root.destroy()

            if selected:
                return Path(selected)
        except Exception:
            pass

    raise FileNotFoundError(
        "Не удалось автоматически найти архив. "
        "Укажите путь в переменной ARCHIVE_PATH в начале этой ячейки и запустите её ещё раз."
    )


def wait_for_dataset(base_dir, timeout_seconds=15):
    # extractall() работает синхронно. Небольшой polling нужен только для сред,
    # где файловая система смонтирована по сети и новые файлы видны не мгновенно.
    deadline = time.time() + timeout_seconds

    while time.time() < deadline:
        dataset_root = find_dataset_root(base_dir)
        if dataset_root is not None:
            return dataset_root
        time.sleep(0.2)

    return None


DATASET_ROOT = find_dataset_root(PROJECT_DIR)

if DATASET_ROOT is None:
    archive_path = choose_archive(PROJECT_DIR)
    extract_dir = PROJECT_DIR / archive_path.stem

    print(f"Распаковываю {archive_path.name} ...")
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive_path, "r") as archive:
        archive.extractall(extract_dir)

    DATASET_ROOT = wait_for_dataset(PROJECT_DIR)

if DATASET_ROOT is None:
    raise RuntimeError("Архив распакован, но ожидаемая структура данных не найдена.")

DATA_DIR = DATASET_ROOT / "data"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
EVENTS_PATH = DATA_DIR / "events.csv.gz"
SAMPLE_SUBMISSION_PATH = DATASET_ROOT / "sample_submission.csv"
METRIC_PATH = DATASET_ROOT / "metric.py"
OUTPUT_DIR = PROJECT_DIR

print("Данные найдены:", DATASET_ROOT)


Распаковываю bot_detection_challenge.zip ...
Данные найдены: /content/bot_detection_challenge


In [ ]:
# 3. Загрузка и базовые проверки

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
events = pd.read_csv(EVENTS_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("train:", train.shape)
print("test:", test.shape)
print("events:", events.shape)
print("sample_submission:", sample_submission.shape)

target_stats = train["target"].value_counts().sort_index().to_frame("count")
target_stats["share"] = target_stats["count"] / len(train)
display(target_stats)

checks = pd.Series(
    {
        "train cookie_id duplicates": train["cookie_id"].duplicated().sum(),
        "test cookie_id duplicates": test["cookie_id"].duplicated().sum(),
        "exact event duplicates": events.duplicated().sum(),
        "train cookies": train["cookie_id"].nunique(),
        "test cookies": test["cookie_id"].nunique(),
        "event cookies": events["cookie_id"].nunique(),
    }
)
display(checks.to_frame("value"))

assert train["cookie_id"].is_unique
assert test["cookie_id"].is_unique


train: (11091, 5)
test: (4909, 4)
events: (328905, 14)
sample_submission: (4909, 2)


,count,share
target,,
0,10192,0.918943
1,899,0.081057


,value
train cookie_id duplicates,0
test cookie_id duplicates,0
exact event duplicates,4863
train cookies,11091
test cookies,4909
event cookies,16000


## Временное окно

Здесь легко получить очень хороший, но неправильный результат. В `events.csv.gz` есть события, которые произошли уже после `window_end_ts`. Их нельзя использовать: на момент предсказания модель этих событий ещё не видела.

Поэтому для каждой cookie оставляю только:

`window_start_ts <= event_ts < window_end_ts`.

Отдельно ниже показываю, почему это не формальность: `captcha_shown` встречается после окна и сильно связан с положительным классом. Если агрегировать всю историю целиком, это будет leakage.


In [ ]:
# 4. Время и фильтрация событий

for frame in (train, test):
    frame["cookie_created_at"] = pd.to_datetime(frame["cookie_created_at"])
    frame["window_start_ts"] = pd.to_datetime(frame["window_start_ts"])
    frame["window_end_ts"] = pd.to_datetime(frame["window_end_ts"])

events["event_ts"] = pd.to_datetime(events["event_ts"])

windows = pd.concat(
    [
        train[["cookie_id", "window_start_ts", "window_end_ts"]],
        test[["cookie_id", "window_start_ts", "window_end_ts"]],
    ],
    ignore_index=True,
)

events_with_window = events.merge(
    windows,
    on="cookie_id",
    how="left",
    validate="many_to_one",
)

inside_window = (
    (events_with_window["event_ts"] >= events_with_window["window_start_ts"])
    & (events_with_window["event_ts"] < events_with_window["window_end_ts"])
)

events_valid = events_with_window.loc[inside_window].copy()
events_valid = events_valid.drop(columns=["window_start_ts", "window_end_ts"])

print("Всего событий:", len(events_with_window))
print("Внутри разрешённого окна:", int(inside_window.sum()))
print("Вне окна:", int((~inside_window).sum()))
print("Полных дублей внутри окна:", events_valid.duplicated().sum())

# Проверяем конкретный источник leakage.
outside_events = events_with_window.loc[~inside_window].copy()
outside_train = outside_events.merge(
    train[["cookie_id", "target"]],
    on="cookie_id",
    how="inner",
    validate="many_to_one",
)

captcha_outside = outside_train.loc[outside_train["event_name"].eq("captcha_shown")]

print("\ncaptcha_shown внутри окна:", events_valid["event_name"].eq("captcha_shown").sum())
print("captcha_shown после окна по классам:")
display(captcha_outside.groupby("target").size().rename("events").to_frame())

assert events_valid["event_name"].eq("captcha_shown").sum() == 0


Всего событий: 328905
Внутри разрешённого окна: 288126
Вне окна: 40779
Полных дублей внутри окна: 4293

captcha_shown внутри окна: 0
captcha_shown после окна по классам:


,events
target,
0,958
1,6970


In [ ]:
# 5. Официальная метрика из задания

metric_spec = importlib.util.spec_from_file_location("official_metric", METRIC_PATH)
metric_module = importlib.util.module_from_spec(metric_spec)
metric_spec.loader.exec_module(metric_module)

precision_at_recall = metric_module.precision_at_recall
recall_at_fpr = metric_module.recall_at_fpr

# Небольшая проверка, что загрузилась именно ожидаемая логика метрики.
assert precision_at_recall([1, 0, 1, 1, 1], [5, 4, 3, 2, 1]) == 0.8

print("Официальная metric.py подключена")


Официальная metric.py подключена


## Признаки

Все признаки считаются на уровне `cookie_id`.

Что использую:
- объём активности и число разных типов событий;
- число уникальных объявлений, категорий, локаций, продавцов и запросов;
- доли конкретных типов действий;
- интервалы между соседними событиями и долю очень коротких интервалов;
- активные часы и минуты;
- статистики `search_page`;
- pointer-поведение: разброс координат и расстояние между соседними точками;
- энтропию событий, категорий, локаций и запросов;
- возраст cookie к началу окна.

С полными дублями есть нюанс. Я не удаляю их из данных целиком: повторяемость сама может быть частью поведения. Поэтому число дублей остаётся отдельным признаком. Но для интервалов между событиями и pointer-траектории использую уникальные строки, иначе технический дубль искусственно создаёт нулевой временной gap.


In [ ]:
# 6. Построение признаков

def entropy_from_counts(counts):
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()

    if total == 0:
        return 0.0

    probabilities = counts[counts > 0] / total
    return float(-(probabilities * np.log(probabilities)).sum())


def safe_feature_value(value):
    text = str(value).lower()
    return re.sub(r"[^0-9a-zA-Z_]+", "_", text).strip("_")


def add_categorical_counts(features, event_frame, column, prefix, event_count):
    count_table = pd.crosstab(event_frame["cookie_id"], event_frame[column])
    count_table = count_table.reindex(features.index, fill_value=0)

    for value in count_table.columns:
        value_name = safe_feature_value(value)
        counts = count_table[value].astype(float)

        features[f"{prefix}_{value_name}_count"] = counts
        features[f"{prefix}_{value_name}_ratio"] = counts / event_count


def build_features(event_frame, cookie_meta):
    events_work = event_frame.copy()

    # У разных источников встречаются технически разные обозначения одной платформы.
    platform = events_work["platform"].fillna("unknown").astype(str)
    platform = platform.str.lower()
    events_work["platform_norm"] = platform.replace({"iphone": "ios"})

    original_event_columns = event_frame.columns.tolist()

    # Дубли не выбрасываем полностью: их количество сохраняем как отдельный сигнал.
    events_work["is_exact_duplicate"] = events_work.duplicated(
        subset=original_event_columns,
        keep="first",
    ).astype("int8")

    # Для последовательностных признаков один и тот же event не должен считаться дважды.
    unique_events = events_work.drop_duplicates(
        subset=original_event_columns,
        keep="first",
    ).copy()

    unique_events = unique_events.sort_values(
        ["cookie_id", "event_ts", "eid"],
        kind="mergesort",
    )

    event_group = events_work.groupby("cookie_id", sort=False)
    unique_group = unique_events.groupby("cookie_id", sort=False)

    features = pd.DataFrame(
        index=pd.Index(cookie_meta["cookie_id"], name="cookie_id")
    )

    # Общий объём и разнообразие активности.
    features["n_events_raw"] = event_group.size()
    features["n_events_unique"] = unique_group.size()
    features["n_exact_duplicates"] = event_group["is_exact_duplicate"].sum()

    event_count = features["n_events_raw"].clip(lower=1)
    features["duplicate_ratio"] = features["n_exact_duplicates"] / event_count

    features["n_event_types"] = event_group["event_name"].nunique()
    features["n_eids"] = event_group["eid"].nunique()
    features["n_platforms"] = event_group["platform_norm"].nunique()
    features["n_items"] = event_group["item_id"].nunique()
    features["n_categories"] = event_group["item_category"].nunique()
    features["n_locations"] = event_group["item_location"].nunique()
    features["n_seller_types"] = event_group["seller_type"].nunique()
    features["n_queries"] = event_group["search_query"].nunique()

    # Отсутствие поля тоже несёт информацию. Presence-ratio позволяет отличить
    # реальный ноль координаты от ситуации, когда pointer-событий вообще не было.
    presence_columns = [
        "item_id",
        "item_category",
        "item_location",
        "seller_type",
        "search_query",
        "search_page",
        "pointer_x",
        "pointer_y",
    ]

    for column in presence_columns:
        filled_values = event_group[column].count()
        features[f"{column}_present_ratio"] = filled_values / event_count

    # Категории превращаем не в raw one-hot на уровне событий, а в число и долю
    # каждого типа действия внутри cookie.
    add_categorical_counts(
        features,
        events_work,
        column="event_name",
        prefix="event",
        event_count=event_count,
    )
    add_categorical_counts(
        features,
        events_work,
        column="platform_norm",
        prefix="platform",
        event_count=event_count,
    )
    add_categorical_counts(
        features,
        events_work,
        column="seller_type",
        prefix="seller",
        event_count=event_count,
    )

    # Числовые поля.
    numeric_columns = ["search_page", "pointer_x", "pointer_y"]
    numeric_stats = ["mean", "std", "min", "max", "median"]

    for column in numeric_columns:
        stats = event_group[column].agg(numeric_stats)

        for statistic in numeric_stats:
            features[f"{column}_{statistic}"] = stats[statistic]

    features["pointer_x_range"] = features["pointer_x_max"] - features["pointer_x_min"]
    features["pointer_y_range"] = features["pointer_y_max"] - features["pointer_y_min"]

    # Относительные признаки устойчивее простых счётчиков при разной активности cookie.
    features["events_per_item"] = event_count / features["n_items"].clip(lower=1)
    features["events_per_category"] = event_count / features["n_categories"].clip(lower=1)
    features["events_per_location"] = event_count / features["n_locations"].clip(lower=1)
    features["locations_per_category"] = (
        features["n_locations"] / features["n_categories"].clip(lower=1)
    )

    # Последовательность событий: интервалы и повторение одного действия подряд.
    sequence = unique_events[
        ["cookie_id", "event_ts", "event_name", "pointer_x", "pointer_y"]
    ].copy()

    sequence["gap_sec"] = (
        sequence.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()
    )

    previous_event = sequence.groupby("cookie_id")["event_name"].shift()
    sequence["same_event_as_prev"] = sequence["event_name"].eq(previous_event).astype(float)

    previous_x = sequence.groupby("cookie_id")["pointer_x"].shift()
    previous_y = sequence.groupby("cookie_id")["pointer_y"].shift()

    sequence["pointer_step"] = np.sqrt(
        (sequence["pointer_x"] - previous_x) ** 2
        + (sequence["pointer_y"] - previous_y) ** 2
    )

    sequence_group = sequence.groupby("cookie_id", sort=False)

    gap_stats = sequence_group["gap_sec"].agg(["mean", "std", "min", "max", "median"])
    for statistic in gap_stats.columns:
        features[f"gap_sec_{statistic}"] = gap_stats[statistic]

    for quantile in (0.10, 0.25, 0.75, 0.90):
        name = f"gap_sec_q{int(quantile * 100)}"
        features[name] = sequence_group["gap_sec"].quantile(quantile)

    for threshold in (5, 10, 30, 60, 300, 900):
        short_gap = sequence["gap_sec"].le(threshold)
        features[f"gap_le_{threshold}_ratio"] = (
            short_gap.groupby(sequence["cookie_id"]).mean()
        )

    active_span = unique_group["event_ts"].max() - unique_group["event_ts"].min()
    features["active_span_sec"] = active_span.dt.total_seconds()
    features["events_per_active_min"] = (
        features["n_events_unique"]
        / (features["active_span_sec"] / 60).clip(lower=1)
    )

    features["same_event_consecutive_ratio"] = sequence_group["same_event_as_prev"].mean()

    pointer_stats = sequence_group["pointer_step"].agg(
        ["mean", "std", "min", "max", "median"]
    )
    for statistic in pointer_stats.columns:
        features[f"pointer_step_{statistic}"] = pointer_stats[statistic]

    # Сколько отдельных минут/часов cookie реально была активна.
    unique_events["event_hour"] = unique_events["event_ts"].dt.floor("h")
    unique_events["event_minute"] = unique_events["event_ts"].dt.floor("min")

    active_hours = unique_events.groupby("cookie_id")["event_hour"].nunique()
    active_minutes = unique_events.groupby("cookie_id")["event_minute"].nunique()

    features["active_hours"] = active_hours
    features["active_minutes"] = active_minutes
    features["events_per_active_hour"] = (
        features["n_events_unique"] / features["active_hours"].clip(lower=1)
    )
    features["events_per_active_minute_bucket"] = (
        features["n_events_unique"] / features["active_minutes"].clip(lower=1)
    )

    # Энтропия показывает, насколько активность сосредоточена в одном значении
    # или распределена по многим событиям/категориям/локациям/запросам.
    entropy_columns = [
        ("event_name", "event"),
        ("item_category", "category"),
        ("item_location", "location"),
        ("search_query", "query"),
    ]

    for column, prefix in entropy_columns:
        non_missing = unique_events.dropna(subset=[column])
        value_counts = non_missing.groupby(["cookie_id", column]).size()

        entropy = value_counts.groupby(level=0).apply(
            lambda values: entropy_from_counts(values.to_numpy())
        )
        features[f"{prefix}_entropy"] = entropy

    # Метаданные cookie доступны до конца окна и не создают leakage.
    meta = cookie_meta.set_index("cookie_id")

    cookie_age = meta["window_start_ts"] - meta["cookie_created_at"]
    features["cookie_age_hours"] = cookie_age.dt.total_seconds() / 3600
    features["window_dow"] = meta["window_start_ts"].dt.dayofweek.astype(float)
    features["cookie_created_hour"] = meta["cookie_created_at"].dt.hour.astype(float)

    # Для агрегатов без наблюдений NaN означает "событий такого типа не было".
    # Presence-ratio выше сохраняет информацию о самом факте отсутствия.
    features = features.replace([np.inf, -np.inf], np.nan)
    features = features.fillna(0)

    return features.reset_index()


In [ ]:
# 7. Матрицы train/test

all_meta = pd.concat(
    [train.drop(columns="target"), test],
    ignore_index=True,
)

all_features = build_features(events_valid, all_meta)

X_train = train[["cookie_id"]].merge(
    all_features,
    on="cookie_id",
    how="left",
    validate="one_to_one",
)

X_test = test[["cookie_id"]].merge(
    all_features,
    on="cookie_id",
    how="left",
    validate="one_to_one",
)

FINAL_FEATURES = [column for column in X_train.columns if column != "cookie_id"]
y = train["target"].to_numpy()

assert len(X_train) == len(train)
assert len(X_test) == len(test)
assert X_train["cookie_id"].is_unique
assert X_test["cookie_id"].is_unique
assert X_train[FINAL_FEATURES].isna().sum().sum() == 0
assert X_test[FINAL_FEATURES].isna().sum().sum() == 0

print("Признаков в финальной модели:", len(FINAL_FEATURES))


Признаков в финальной модели: 105


## Валидация

Test расположен позже train, поэтому случайный `train_test_split` здесь был бы слишком удобным и не очень честным. Я имитирую реальное использование модели: учусь на прошлом и проверяюсь на следующих днях.

Для основного контроля беру последние 7 дней train как out-of-time holdout — это близко к длине тестового периода. До финальной модели также проверял сдвиг границы holdout, User-Agent признаки, более широкий набор агрегатов и несколько конфигураций CatBoost.


In [ ]:
# 8. Простой baseline

unique_dates = sorted(train["window_start_ts"].dt.normalize().unique())
baseline_days = min(3, max(1, len(unique_dates) - 1))
baseline_start = pd.Timestamp(unique_dates[-baseline_days])

baseline_valid = train["window_start_ts"].ge(baseline_start).to_numpy()
baseline_train = ~baseline_valid

baseline_features = ["n_events_raw", "n_items"]

baseline_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=SEED,
    n_jobs=-1,
)

baseline_model.fit(
    X_train.loc[baseline_train, baseline_features],
    y[baseline_train],
)

baseline_score = baseline_model.predict_proba(
    X_train.loc[baseline_valid, baseline_features]
)[:, 1]

print(
    "Baseline P@R>=0.70:",
    round(precision_at_recall(y[baseline_valid], baseline_score), 4),
)
print(
    "Baseline ROC-AUC:",
    round(roc_auc_score(y[baseline_valid], baseline_score), 4),
)


Baseline P@R>=0.70: 0.1062
Baseline ROC-AUC: 0.6081


In [ ]:
# 9. Финальная конфигурация и out-of-time проверка

FINAL_ITERATIONS = 600
FINAL_SEEDS = [17, 42, 137, 777, 2026]
VALIDATION_SEEDS = [42]


def make_model(seed):
    return CatBoostClassifier(
        iterations=FINAL_ITERATIONS,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        l2_leaf_reg=5,
        random_strength=0.8,
        random_seed=seed,
        verbose=False,
        thread_count=-1,
    )


def fit_ensemble(X_fit, y_fit, X_score, seeds):
    models = []
    predictions = []

    for seed in seeds:
        model = make_model(seed)
        model.fit(X_fit, y_fit, verbose=False)

        models.append(model)
        predictions.append(model.predict_proba(X_score)[:, 1])

    return models, predictions


validation_days = min(7, max(1, len(unique_dates) - 1))
validation_start = pd.Timestamp(unique_dates[-validation_days])

validation_mask = train["window_start_ts"].ge(validation_start).to_numpy()
development_mask = ~validation_mask

validation_models, validation_predictions = fit_ensemble(
    X_train.loc[development_mask, FINAL_FEATURES],
    y[development_mask],
    X_train.loc[validation_mask, FINAL_FEATURES],
    VALIDATION_SEEDS,
)

validation_score = np.mean(validation_predictions, axis=0)

validation_result = pd.Series(
    {
        "train_rows": int(development_mask.sum()),
        "validation_rows": int(validation_mask.sum()),
        "validation_bots": int(y[validation_mask].sum()),
        "P@R>=0.70": precision_at_recall(y[validation_mask], validation_score),
        "ROC-AUC": roc_auc_score(y[validation_mask], validation_score),
    }
)

display(validation_result.to_frame("value"))


,value
train_rows,5930.000000
validation_rows,5161.000000
validation_bots,408.000000
P@R>=0.70,0.754617
ROC-AUC,0.937350


### Что проверял перед финальным выбором

В черновом исследовании я пробовал несколько вариантов, но в финальный notebook не тащу десятки почти одинаковых обучений.

Коротко по результатам:

| Что менял | Что получилось |
|---|---|
| Простой baseline | `P@R>=0.70 ≈ 0.11` |
| User-Agent признаки | местами помогали, но результат заметно гулял по времени; исключил |
| Расширение до ~150 признаков | на коротких фолдах иногда лучше, на длинных holdout хуже; оставил компактный набор |
| CatBoost depth 7 | ROC-AUC почти тот же, по целевой метрике менее стабилен |
| 400 / 500 / 600 итераций | 600 дало лучший средний результат на длинных holdout |
| Несколько seed | уменьшили зависимость результата от случайности |

На нескольких длинных временных holdout итоговая схема держалась примерно в диапазоне `0.70–0.78` по основной метрике. Поэтому финальный выбор делал не по одному самому красивому числу, а по устойчивости во времени.


In [ ]:
# 10. Финальное обучение на всём train

final_models, test_predictions = fit_ensemble(
    X_train[FINAL_FEATURES],
    y,
    X_test[FINAL_FEATURES],
    FINAL_SEEDS,
)

# Усредняем вероятности. Для этой метрики важен порядок объектов по score,
# а несколько seed немного снижают variance одной конкретной модели.
final_test_score = np.mean(test_predictions, axis=0)

print("score min:", final_test_score.min())
print("score mean:", final_test_score.mean())
print("score max:", final_test_score.max())


score min: 0.00013029290758399895
score mean: 0.08223100123913296
score max: 0.9988657809701728


In [ ]:
# 11. Важность признаков

importance_matrix = np.vstack(
    [model.get_feature_importance() for model in final_models]
)

feature_importance = pd.DataFrame(
    {
        "feature": FINAL_FEATURES,
        "importance": importance_matrix.mean(axis=0),
    }
)

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False,
    ignore_index=True,
)

display(feature_importance.head(25))


,feature,importance
0,gap_le_60_ratio,4.481884
1,events_per_item,4.396621
2,locations_per_category,4.043475
3,gap_le_30_ratio,2.964506
4,gap_sec_median,2.812591
5,cookie_age_hours,2.753580
6,pointer_x_max,2.715890
7,gap_sec_min,2.334976
8,n_categories,2.273966
9,gap_sec_q75,2.176844


In [ ]:
# 12. Submission и финальные проверки

prediction_table = pd.DataFrame(
    {
        "cookie_id": X_test["cookie_id"],
        "score": final_test_score,
    }
)

# Порядок строк берём из sample_submission, а не полагаемся на порядок merge.
submission = sample_submission[["cookie_id"]].merge(
    prediction_table,
    on="cookie_id",
    how="left",
    validate="one_to_one",
)

assert list(submission.columns) == ["cookie_id", "score"]
assert len(submission) == len(test)
assert submission["cookie_id"].is_unique
assert submission["score"].notna().all()
assert np.isfinite(submission["score"]).all()
assert submission["score"].between(0, 1).all()
assert submission["cookie_id"].tolist() == sample_submission["cookie_id"].tolist()
assert set(submission["cookie_id"]) == set(test["cookie_id"])

SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

# Читаем уже записанный файл обратно. Так проверяется именно тот CSV,
# который будет отправлен, а не DataFrame в памяти. (Опционально для моего спокойствия)
submission_check = pd.read_csv(SUBMISSION_PATH)

expected_scores = prediction_table.rename(columns={"score": "expected_score"})
submission_check = submission_check.merge(
    expected_scores,
    on="cookie_id",
    how="left",
    validate="one_to_one",
)

assert submission_check["expected_score"].notna().all()
assert np.allclose(
    submission_check["score"],
    submission_check["expected_score"],
    rtol=0,
    atol=1e-12,
)

file_hash = hashlib.sha256(SUBMISSION_PATH.read_bytes()).hexdigest()

print("Submission готов:", SUBMISSION_PATH)
print("Строк:", len(submission))
print("SHA256:", file_hash)
display(submission.head())


Submission готов: /content/submission.csv
Строк: 4909
SHA256: 022b93d5b2b5fd7b0885480f3726e149a40dc3c2f792c144ded7f0735af0fabc


,cookie_id,score
0,ck_99a4e5ef02d89493,0.014294
1,ck_54d463f7fd89ec3d,0.009272
2,ck_0fbbc7a6af300368,0.001088
3,ck_8fd4937eadd64fb6,0.004729
4,ck_dbb4bd4eda97255d,0.010888


## Итог

Основной прирост качества получился не из-за сложного тюнинга CatBoost, а из-за правильной постановки пайплайна:

1. события после окна наблюдения исключены;
2. признаки собраны на уровне cookie;
3. validation идёт вперёд по времени, как и реальный test;
4. модель сравнивается с простым baseline;
5. нестабильные User-Agent и лишние агрегаты в финальную версию не вошли;
6. финальный score — среднее пяти CatBoost с фиксированными seed.

При повторном запуске на тех же данных notebook заново обучает модели и создаёт `submission.csv` с теми же правилами, параметрами и seed. Версии основных библиотек вынесены в `requirements.txt`.
